# IVModel Mechanistic Interpretability

Visualizes what the trained `IVModel` checkpoints learned: attention patterns, QK circuits, and fixed-lag attention detection (a numeric-time-series analog of GPT-2's induction heads). Runs on both `Pratham007xo/iv-forecast-constituent-124m` and `Pratham007xo/iv-forecast-index-124m`.

## Setup

In [ ]:
!git clone https://github.com/prathamkul007-max/IV_mech_interp_model.git
%cd IV_mech_interp_model
!pip install -q -r requirements.txt
!pip install -q -r Mechanistic-Interpretability/requirements.txt

In [ ]:
import sys
sys.path.insert(0, 'Mechanistic-Interpretability')

import torch
print('CUDA available:', torch.cuda.is_available())

## Prepare validation data (needed as real model input for the visualizations)

In [ ]:
!python scripts/prepare_options_iv.py --output-dir options_iv_data --seq-length 32 --stride 5
!python scripts/prepare_options_iv_index.py --output-dir options_iv_index_data --seq-length 32 --stride 1

## Load both checkpoints from the Hugging Face Hub

In [ ]:
from load_iv_model import load_iv_checkpoint

constituent_model, constituent_scaler = load_iv_checkpoint('Pratham007xo/iv-forecast-constituent-124m')
index_model, index_scaler = load_iv_checkpoint('Pratham007xo/iv-forecast-index-124m')

print('Constituent model:', constituent_model.config)
print('Index model:', index_model.config)

## Verify hooks against both real checkpoints

In [ ]:
from hooks import verify_hooks
from sample_windows import load_sample_windows

constituent_windows = load_sample_windows('options_iv_data/valid.npz', n=200)
index_windows = load_sample_windows('options_iv_index_data/valid.npz', n=200)

constituent_features = torch.from_numpy(constituent_windows).float()
index_features = torch.from_numpy(index_windows).float()

print('Constituent model hook verification:', verify_hooks(constituent_model, constituent_features[:8]))
print('Index model hook verification:', verify_hooks(index_model, index_features[:8]))

## Attention pattern visualization

In [ ]:
from visualize_attention import plot_attention_heatmap_static, plot_attention_heatmap_interactive, plot_all_heads_grid

plot_attention_heatmap_static(constituent_model, constituent_features[:1], layer=0, head=0)
plot_all_heads_grid(constituent_model, constituent_features[:1], layer=constituent_model.config.num_layers - 1)

In [ ]:
fig = plot_attention_heatmap_interactive(constituent_model, constituent_features[:1], layer=0, head=0)
fig.show()

## QK circuit analysis

In [ ]:
from qk_circuit_analysis import plot_qk_interactions_static, plot_qk_interactions_interactive

plot_qk_interactions_static(constituent_model, constituent_features[:8], layer=0)
fig = plot_qk_interactions_interactive(constituent_model, constituent_features[:8], layer=constituent_model.config.num_layers - 1)
fig.show()

## Fixed-lag attention detection

In [ ]:
from fixed_lag_detection import compute_lag_profile, plot_lag_heatmap, plot_layer_scores

constituent_lag_df = compute_lag_profile(constituent_model, constituent_features)
constituent_lag_df.sort_values('peak_score', ascending=False).head(10)

In [ ]:
plot_lag_heatmap(constituent_lag_df)
plot_layer_scores(constituent_lag_df, layer=0)

## Repeat all three analyses for the index-level model

In [ ]:
plot_attention_heatmap_static(index_model, index_features[:1], layer=0, head=0)
plot_qk_interactions_static(index_model, index_features[:8], layer=0)

index_lag_df = compute_lag_profile(index_model, index_features)
plot_lag_heatmap(index_lag_df)
index_lag_df.sort_values('peak_score', ascending=False).head(10)